# 02 Construction and Semantics

**Learning goals**

- Construct AOs from scalar, vector, batched, and ragged xarray datasets.
- Learn the construction contract: ordered samples, independent groups, payload dimensions, and param coordinates.
- Understand safe public data access and fail-closed validation boundaries.


In [1]:
import numpy as np
import xarray as xr
from IPython.display import display

from tal import AnalysisObject

from pathlib import Path
import sys

_release_dir = Path.cwd()
if not (_release_dir / '_helpers.py').exists():
    candidate = Path.cwd() / 'examples' / 'release'
    if candidate.exists():
        _release_dir = candidate
if str(_release_dir) not in sys.path:
    sys.path.append(str(_release_dir))

from _helpers import (
    build_framegraph_spatial_payloads,
    build_topology_edge_payloads,
    compact_dataset,
    display_result,
    explain_expected_failure,
    make_imu_gps_alignment_aos,
    make_kinematics_demo_objects,
    make_pose_demo_objects,
    show_ao,
    summarize_ao,
)


## Start with a scalar trajectory

The simplest AO has one variable over one ordered sequence dimension. Empty `core_dims=()` means each sample carries a scalar payload.


In [2]:
signal_ds = xr.Dataset(
    data_vars={'signal': ('sample', np.sin(np.linspace(0.0, 2.0 * np.pi, 8)))},
    coords={'sample': np.arange(8), 'time_s': ('sample', np.linspace(0.0, 1.0, 8))},
)
signal_ao = AnalysisObject.from_data(
    signal_ds,
    sequence_dim='sample',
    core_dims=(),
    param_coord='time_s',
    validate=True,
)

show_ao('scalar signal AO', signal_ao)


#### scalar signal AO

,field,value
0,type,AnalysisObject
1,dims,{'sample': 8}
2,data_vars,[signal]
3,coords,"[sample, time_s]"
4,batch_dims,()
5,sequence_dim,sample
6,core_dims,()
7,param_coord,time_s
8,sequence_size_coord,None
9,frames,None


<xarray.Dataset> Size: 192B
Dimensions:  (sample: 8)
Coordinates:
  * sample   (sample) int64 64B 0 1 2 3 4 5 6 7
    time_s   (sample) float64 64B 0.0 0.1429 0.2857 0.4286 ... 0.7143 0.8571 1.0
Data variables:
    signal   (sample) float64 64B 0.0 0.7818 0.9749 ... -0.7818 -2.449e-16
Attributes:
    tal:      {'version': 1, 'core': {'roles': {'batch_dims': [], 'core_dims'...

What to notice:

- `sample` is the sequence dimension.
- There are no batch dimensions.
- There are no core payload dimensions.
- `time_s` parameterizes the samples for later interpolation.


## Public data is safe to inspect

`ao.as_dataset()` returns a safe deep snapshot. The explicit `copy="none"` mode returns the backing Dataset for expert ownership crossings, so mutation there is visible.


In [3]:
# Hold one explicit backing reference for the ownership checks below
raw_ref = signal_ao.as_dataset(copy="none")
original_signal = raw_ref['signal'].values.copy()

# Modifying the default deep snapshot should not mutate the backing store
safe_copy = signal_ao.as_dataset()
safe_copy['signal'].values[:] = 0.0
assert not bool((raw_ref['signal'].values == 0.0).all())

# The explicit raw Dataset mode mutates the backing store
raw_ref['signal'].values[:] = 0.0
assert bool((raw_ref['signal'].values == 0.0).all())

# Restore demo values so the rest of the notebook remains readable
raw_ref['signal'].values[:] = original_signal

{
    'deep_snapshot_mutates_backing_store': False,
    'raw_dataset_mutates_backing_store': True,
}


{'deep_snapshot_mutates_backing_store': False,
 'raw_dataset_mutates_backing_store': True}

## Add a vector payload

A vector trajectory still has one sequence dimension. The extra `axis` dimension is payload structure, so it belongs in `core_dims`.


In [4]:
vector_ds = xr.Dataset(
    data_vars={
        'velocity': (
            ('sample', 'axis'),
            np.stack([
                np.ones(8),
                np.sin(np.linspace(0.0, 2.0 * np.pi, 8)),
                np.zeros(8),
            ], axis=-1),
        )
    },
    coords={
        'sample': np.arange(8),
        'axis': ['x', 'y', 'z'],
        'time_s': ('sample', np.linspace(0.0, 1.0, 8)),
    },
)
vector_ao = AnalysisObject.from_data(
    vector_ds,
    sequence_dim='sample',
    core_dims=('axis',),
    param_coord='time_s',
    validate=True,
)

show_ao('vector payload AO', vector_ao)


#### vector payload AO

,field,value
0,type,AnalysisObject
1,dims,"{'sample': 8, 'axis': 3}"
2,data_vars,[velocity]
3,coords,"[sample, axis, time_s]"
4,batch_dims,()
5,sequence_dim,sample
6,core_dims,"(axis,)"
7,param_coord,time_s
8,sequence_size_coord,None
9,frames,None


<xarray.Dataset> Size: 332B
Dimensions:   (sample: 8, axis: 3)
Coordinates:
  * sample    (sample) int64 64B 0 1 2 3 4 5 6 7
  * axis      (axis) <U1 12B 'x' 'y' 'z'
    time_s    (sample) float64 64B 0.0 0.1429 0.2857 ... 0.7143 0.8571 1.0
Data variables:
    velocity  (sample, axis) float64 192B 1.0 0.0 0.0 1.0 ... 1.0 -2.449e-16 0.0
Attributes:
    tal:      {'version': 1, 'core': {'roles': {'batch_dims': [], 'core_dims'...

## Add batches and ragged validity

Batch dimensions represent independent runs. Ragged validity lets TAL distinguish padded storage from real samples.


In [5]:
grouped_ds = xr.Dataset(
    data_vars={
        'position': (
            ('trial', 'sample', 'axis'),
            np.array(
                [
                    [[0.0, 0.0, 0.0], [0.2, 0.0, 0.0], [0.4, 0.1, 0.0], [0.6, 0.1, 0.0]],
                    [[1.0, 0.0, 0.0], [1.2, 0.1, 0.0], [999.0, 999.0, 999.0], [999.0, 999.0, 999.0]],
                ],
                dtype=float,
            ),
        )
    },
    coords={
        'trial': ['A', 'B'],
        'sample': np.arange(4),
        'axis': ['x', 'y', 'z'],
        'time_s': (
            ('trial', 'sample'),
            np.array([[0.0, 0.2, 0.4, 0.6], [0.0, 0.25, 0.5, 0.75]], dtype=float),
        ),
        'group_size': ('trial', np.array([4, 2], dtype=np.int64)),
    },
)

grouped_ao = AnalysisObject.from_data(
    grouped_ds,
    sequence_dim='sample',
    batch_dims=('trial',),
    core_dims=('axis',),
    param_coord='time_s',
    sequence_size_coord='group_size',
    validate=True,
)

show_ao('batched ragged AO', grouped_ao)


#### batched ragged AO

,field,value
0,type,AnalysisObject
1,dims,"{'trial': 2, 'sample': 4, 'axis': 3}"
2,data_vars,[position]
3,coords,"[trial, sample, axis, time_s, group_size]"
4,batch_dims,"(trial,)"
5,sequence_dim,sample
6,core_dims,"(axis,)"
7,param_coord,time_s
8,sequence_size_coord,group_size
9,frames,None


<xarray.Dataset> Size: 324B
Dimensions:     (trial: 2, sample: 4, axis: 3)
Coordinates:
  * trial       (trial) <U1 8B 'A' 'B'
  * sample      (sample) int64 32B 0 1 2 3
  * axis        (axis) <U1 12B 'x' 'y' 'z'
    time_s      (trial, sample) float64 64B 0.0 0.2 0.4 0.6 0.0 0.25 0.5 0.75
    group_size  (trial) int64 16B 4 2
Data variables:
    position    (trial, sample, axis) float64 192B 0.0 0.0 0.0 ... 999.0 999.0
Attributes:
    tal:      {'version': 1, 'core': {'roles': {'batch_dims': ['trial'], 'cor...

## Compare naive and validity-aware reductions

The padded values are intentionally large. xarray alone treats them as data; TAL uses `group_size` as validity metadata.


In [6]:
naive = grouped_ao.as_dataset()['position'].mean(dim='sample')
tal_mean = grouped_ao.mean(dim='sample').as_dataset()['position']
comparison = xr.Dataset({'naive_xarray_mean': naive, 'tal_validity_aware_mean': tal_mean})
display(comparison)


<xarray.Dataset> Size: 132B
Dimensions:                  (trial: 2, axis: 3)
Coordinates:
  * trial                    (trial) <U1 8B 'A' 'B'
  * axis                     (axis) <U1 12B 'x' 'y' 'z'
    group_size               (trial) int64 16B 4 2
Data variables:
    naive_xarray_mean        (trial, axis) float64 48B 0.3 0.05 ... 499.5 499.5
    tal_validity_aware_mean  (trial, axis) float64 48B 0.3 0.05 0.0 1.1 0.05 0.0

What to notice:

- Padding is a storage detail, not valid trajectory content.
- TAL reductions preserve the remaining batch/core semantics.
- Construction is the right place to declare validity because later operations depend on it.


## Common construction mistakes

Validation should fail when declared roles do not match the data. These examples are expected failures.


In [7]:
def invalid_core_dim():
    AnalysisObject.from_data(
        vector_ds,
        sequence_dim='sample',
        core_dims=('missing_axis',),
        param_coord='time_s',
        validate=True,
    )


def bad_param_coord():
    AnalysisObject.from_data(
        signal_ds,
        sequence_dim='sample',
        core_dims=(),
        param_coord='missing_time',
        validate=True,
    )


explain_expected_failure(invalid_core_dim, 'invalid core dim')
explain_expected_failure(bad_param_coord, 'invalid param coordinate')


invalid core dim: expected SchemaError: [schema.roles.core_dims.not_in_dataset] at tal.core.roles.core_dims: expected='all dims in ds.dims', actual={'missing': ['missing_axis'], 'ds_dims': ['sample', 'axis']}. Fix: use only existing dataset dims in core_dims
invalid param coordinate: expected SchemaError: [schema.param_coord.not_found] at tal.core.param_coord.name: expected='coord in ds.coords', actual={'coord': 'missing_time', 'coords': ['sample', 'time_s']}. Fix: add the coordinate or choose an existing coord name


## Takeaways

- Construction answers: what is ordered, what is independent, what is payload, and what coordinate parameterizes samples?
- Scalar payloads have empty `core_dims`.
- Vector or matrix payload axes belong in `core_dims`.
- Ragged validity makes padded dense storage analytically safe.

**Exercise:** add a `sensor` batch dimension to the vector example and update `batch_dims` accordingly.
